In [1]:
!pip install confluent_kafka
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 14.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 38.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 MB 23.5 MB/s eta 0:00:0000:0100:01


In [1]:
from confluent_kafka import Consumer, KafkaError, KafkaException
import sys
import io

conf = {
    "bootstrap.servers": "broker:9092",
    "group.id": "foo",
    "auto.offset.reset": "smallest",
}
running = True
cons = Consumer(conf)
messages = list()

def consume_loop(consumer, topics):
    
    try:
        for i in range(23003):
            consumer.subscribe(topics)
        
            msg_count = 0
            #while running:
            msg = consumer.poll(timeout=10.0)
            # if msg is None:
            #     continue
        
            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    # End of partition event
                    sys.stderr.write(
                        "%% %s [%d] reached end at offset %d\n"
                        % (msg.topic(), msg.partition(), msg.offset())
                    )
                elif msg.error():
                    raise KafkaException(msg.error())
            else:
                # data = io.StringIO(msg.value().decode("utf-8"))
                # print(data.readline())
                data = msg.value().decode("utf-8")
                messages.append(data)
                msg_count += 1
                if msg_count > 1:
                    consumer.commit(asynchronous=False)
    finally:
        # Close down consumer to commit final offsets.
        consumer.close()


consume_loop(cons, ["test"])
print(len(messages))

23003


In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
# Select columns to train on

messageList = list()
for message in messages:
    messageDict = dict()
    lines = message.strip().split("\n")
    for line in lines:
        key, value = line.split(maxsplit=1)
        messageDict[key] = value
    messageList.append(messageDict)
    

# Split data into train/test sets
df = pd.DataFrame(messageList, columns=messageList[0].keys())
X = df[["resp_pkts", "orig_ip_bytes", "missed_bytes", "duration", "orig_pkts", "resp_ip_bytes", "dest_port", "orig_bytes",
         "resp_bytes", "src_port", "ts"]].dropna()
Y = df["mitre_attack_tactics"].apply(lambda y: 'non-malicious' if y == 'none' else 'malicious')
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=1)
print(X_train.head())
print(Y_train.head())


      resp_pkts orig_ip_bytes missed_bytes  duration orig_pkts resp_ip_bytes  \
18909         0           138            0  0.000125         2             0   
7918          0           134            0  5.005172         2             0   
12155         0            56            0       NaN         1             0   
12972         0           134            0   0.00001         2             0   
8605          2           186            0  0.002333         2           186   

      dest_port orig_bytes resp_bytes src_port                 ts  
18909        53       82.0        0.0    57992  1642151221.271254  
7918         53       78.0        0.0    36635   1644465519.99574  
12155        53        NaN        NaN    11541   1641815697.53028  
12972        53       78.0        0.0    53673  1641708823.703983  
8605         53      130.0      130.0    52915    1644465540.0189  
18909    non-malicious
7918         malicious
12155    non-malicious
12972    non-malicious
8605         malici

In [42]:
# for col in X_train.columns:
#    print(f"Col: {col}   \t\tHas NaN value: {X_train.isna()}")
print(df.isna().any())
print(Y.unique())
print(len(Y))

resp_pkts               False
service                 False
orig_ip_bytes           False
local_resp              False
missed_bytes            False
protocol                False
duration                False
conn_state              False
dest_ip                 False
orig_pkts               False
community_id            False
resp_ip_bytes           False
dest_port               False
orig_bytes              False
local_orig              False
datetime                False
history                 False
resp_bytes              False
uid                     False
src_port                False
ts                      False
src_ip                  False
mitre_attack_tactics    False
Name:                   False
dtype: bool
['non-malicious' 'malicious']
23003


In [12]:
from sklearn.ensemble import IsolationForest

# params
#
#   n_estimators: # of base estimators in ensemble
#   contamination: Percentage of outliers in dataset
#   random_state: seed for model
model = IsolationForest(n_estimators=100, contamination=0.5, random_state=1)

model.fit(X_train)
predictions = model.predict(X_test)

anomaly_score = model.decision_function(X_test)

print(f"Labels: {Y_train}")
print(f"Predictions: {predictions}")
print(f"Anom Score: {anomaly_score}")

ValueError: Input X contains NaN.
IsolationForest does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values